In [1]:
!pip install streamlit plotly pandas -q
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
-O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 93.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 105.1 MB/s eta 0:00:00


In [2]:
%%writefile app.py
import streamlit as st
import pandas as pd
import plotly.express as px

st.set_page_config(
    page_title="Film Production Dashboard",
    page_icon="🎬",
    layout="wide"
)

st.title("🎬 Film Production Dashboard")
st.write("Analyze movie budgets, revenues, genres, and ROI.")

# -----------------------------
# Sample movie data
# -----------------------------
default_movies = [
    {
        "Title": "Titanic",
        "Director": "James Cameron",
        "Year": 1997,
        "Genre": "Romance",
        "Budget": 200_000_000,
        "Revenue": 2_264_000_000
    },
    {
        "Title": "Avatar",
        "Director": "James Cameron",
        "Year": 2009,
        "Genre": "Sci-Fi",
        "Budget": 237_000_000,
        "Revenue": 2_923_000_000
    },
    {
        "Title": "The Godfather",
        "Director": "Francis Ford Coppola",
        "Year": 1972,
        "Genre": "Crime",
        "Budget": 6_000_000,
        "Revenue": 250_000_000
    },
    {
        "Title": "Parasite",
        "Director": "Bong Joon-ho",
        "Year": 2019,
        "Genre": "Thriller",
        "Budget": 11_400_000,
        "Revenue": 263_000_000
    },
    {
        "Title": "The Dark Knight",
        "Director": "Christopher Nolan",
        "Year": 2008,
        "Genre": "Action",
        "Budget": 185_000_000,
        "Revenue": 1_006_000_000
    }
]

if "movies" not in st.session_state:
    st.session_state.movies = pd.DataFrame(default_movies)

# -----------------------------
# Functions
# -----------------------------
def calculate_roi(budget, revenue):
    if budget == 0:
        return 0
    return ((revenue - budget) / budget) * 100

def get_decade(year):
    return f"{year // 10 * 10}s"

def get_result_label(roi):
    if roi >= 300:
        return "Blockbuster"
    elif roi >= 100:
        return "Successful"
    elif roi >= 0:
        return "Moderate"
    else:
        return "Flop"

# -----------------------------
# Sidebar: Add movie
# -----------------------------
st.sidebar.header("➕ Add a Movie")

with st.sidebar.form("add_movie_form"):
    title = st.text_input("Title")
    director = st.text_input("Director")
    year = st.number_input("Year", min_value=1900, max_value=2035, value=2024)
    genre = st.selectbox(
        "Genre",
        [
            "Action",
            "Drama",
            "Comedy",
            "Romance",
            "Sci-Fi",
            "Fantasy",
            "Thriller",
            "Crime",
            "Horror",
            "Animation",
            "Documentary",
            "Other"
        ]
    )
    budget = st.number_input("Budget ($)", min_value=0, value=10_000_000, step=1_000_000)
    revenue = st.number_input("Revenue ($)", min_value=0, value=50_000_000, step=1_000_000)

    submitted = st.form_submit_button("Add Movie")

    if submitted:
        if title.strip() == "" or director.strip() == "":
            st.sidebar.error("Please enter both title and director.")
        else:
            new_movie = pd.DataFrame([
                {
                    "Title": title,
                    "Director": director,
                    "Year": int(year),
                    "Genre": genre,
                    "Budget": budget,
                    "Revenue": revenue
                }
            ])

            st.session_state.movies = pd.concat(
                [st.session_state.movies, new_movie],
                ignore_index=True
            )

            st.sidebar.success(f"{title} added successfully!")

# -----------------------------
# Prepare data
# -----------------------------
df = st.session_state.movies.copy()
df["ROI (%)"] = df.apply(lambda row: calculate_roi(row["Budget"], row["Revenue"]), axis=1)
df["Profit"] = df["Revenue"] - df["Budget"]
df["Decade"] = df["Year"].apply(get_decade)
df["Result"] = df["ROI (%)"].apply(get_result_label)

# -----------------------------
# Filter
# -----------------------------
st.subheader("🎞️ Filter by Decade")

decades = sorted(df["Decade"].unique())
selected_decades = st.multiselect(
    "Select decade(s)",
    options=decades,
    default=decades
)

filtered_df = df[df["Decade"].isin(selected_decades)]

# -----------------------------
# Metrics
# -----------------------------
st.subheader("📌 Key Numbers")

total_budget = filtered_df["Budget"].sum()
total_revenue = filtered_df["Revenue"].sum()
total_profit = filtered_df["Profit"].sum()
average_roi = filtered_df["ROI (%)"].mean() if not filtered_df.empty else 0

col1, col2, col3, col4 = st.columns(4)

col1.metric("Total Budget", f"${total_budget:,.0f}")
col2.metric("Total Revenue", f"${total_revenue:,.0f}")
col3.metric("Total Profit", f"${total_profit:,.0f}")
col4.metric("Average ROI", f"{average_roi:.2f}%")

# -----------------------------
# Table
# -----------------------------
st.subheader("📋 Movie Data")

display_df = filtered_df.copy()
display_df["Budget"] = display_df["Budget"].map("${:,.0f}".format)
display_df["Revenue"] = display_df["Revenue"].map("${:,.0f}".format)
display_df["Profit"] = display_df["Profit"].map("${:,.0f}".format)
display_df["ROI (%)"] = display_df["ROI (%)"].map("{:.2f}%".format)

st.dataframe(display_df, use_container_width=True)

# -----------------------------
# Charts
# -----------------------------
if filtered_df.empty:
    st.warning("No movies match the selected decade filter.")
else:
    st.subheader("📊 Budget vs Revenue")

    budget_revenue_df = filtered_df.melt(
        id_vars="Title",
        value_vars=["Budget", "Revenue"],
        var_name="Category",
        value_name="Amount"
    )

    bar_chart = px.bar(
        budget_revenue_df,
        x="Title",
        y="Amount",
        color="Category",
        barmode="group",
        title="Budget vs Revenue by Movie",
        labels={"Amount": "Amount ($)", "Title": "Movie"}
    )

    st.plotly_chart(bar_chart, use_container_width=True)

    st.subheader("🥧 Genre Distribution")

    genre_counts = filtered_df["Genre"].value_counts().reset_index()
    genre_counts.columns = ["Genre", "Count"]

    pie_chart = px.pie(
        genre_counts,
        names="Genre",
        values="Count",
        title="Movie Genre Distribution"
    )

    st.plotly_chart(pie_chart, use_container_width=True)

    st.subheader("🏆 Top Movies by ROI")

    top_roi_df = filtered_df.sort_values(by="ROI (%)", ascending=False)
    st.dataframe(
        top_roi_df[
            ["Title", "Director", "Year", "Genre", "Budget", "Revenue", "Profit", "ROI (%)", "Result"]
        ],
        use_container_width=True
    )

# -----------------------------
# Reset
# -----------------------------
st.divider()

if st.button("Reset to Default Movies"):
    st.session_state.movies = pd.DataFrame(default_movies)
    st.success("Dashboard reset to default movie data.")
    st.rerun()

Writing app.py


In [3]:
!streamlit run app.py \
--server.headless true \
--server.enableCORS false \
--server.enableXsrfProtection false &>/dev/null &

import time
time.sleep(3)

!cloudflared tunnel --url localhost:8501

2026-06-10T02:50:51Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-10T02:50:51Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-10T02:50:57Z INF +--------------------------------------------------------------------------------------------+
2026-06-10T02:50:57Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-10T02:50:57Z INF |  https://winners-leu-personalized-declared.trycloudfla

In [ ]:
!pkill -f streamlit

!streamlit run app.py \
--server.headless true \
--server.port 8501 \
--server.enableCORS false \
--server.enableXsrfProtection false &>/dev/null &

import time
time.sleep(5)

!cloudflared tunnel --url http://localhost:8501

2026-06-10T02:54:49Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-10T02:54:49Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-10T02:54:54Z INF +--------------------------------------------------------------------------------------------+
2026-06-10T02:54:54Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-10T02:54:54Z INF |  https://contrast-pregnant-satellite-something.tryclou